# Metadata

Domain: Mathematics

Subdomain: Numerical Root-Finding

References:

- C. T. Kelley, *Iterative Methods for Linear and Nonlinear Equations*, SIAM, 1995, Ch. 2-3. https://doi.org/10.1137/1.9781611970920
- J. Nocedal & S. J. Wright, *Numerical Optimization*, 2nd ed., Springer, 2006, Ch. 11 (Nonlinear Least Squares). https://doi.org/10.1007/978-0-387-40065-5

Required Dependencies: numpy (standard numerical library used by all reference solutions and tests). Testing: pytest, provided by the standard project environment.

# Title

Solving Systems of Nonlinear Equations with Newton's Method


# Subproblem 1

## Prompt

Implement a **central-difference** approximation of the Jacobian matrix of a vector-valued function $F: \mathbb{R}^n \to \mathbb{R}^m$. To stay numerically resolvable in floating point, the per-coordinate perturbation must be **scaled by that coordinate's own magnitude** rather than using the raw `eps` unscaled (see Background for the exact formula and why a fixed, unscaled step silently breaks for large $|x_j|$).

```python
def finite_difference_jacobian(F: Callable, x: np.ndarray, eps: float = 1e-6) -> np.ndarray:
    """
    Parameters
    ----------
    F : callable
        F(x) -> ndarray, shape (m,).
    x : ndarray, shape (n,)
        Point at which to evaluate the Jacobian.
    eps : float
        Nominal (relative) finite-difference step size.

    Returns
    -------
    J : ndarray, shape (m, n)
        Approximate Jacobian, J[i, j] = dF_i/dx_j, computed via a
        symmetric (central) difference with per-coordinate step
        h_j = eps * max(1, |x_j|). When x has shape (0,), the returned
        array has shape (m, 0), where m = len(F(x)).

    Raises
    ------
    ValueError
        If eps <= 0, if x is not a 1-D array, or if F(x) does not
        return a 1-D array.
    """
    ...
    return J
```


## Background

The Jacobian of $F=(F_1,\dots,F_m)$ at $x=(x_1,\dots,x_n)$ is the $m\times n$ matrix of partial derivatives $J_{ij}=\partial F_i/\partial x_j$. Each column $j$ can be estimated independently by perturbing only coordinate $j$ in both directions and taking a **central** (symmetric) difference,

$$
J_{:,j} \;\approx\; \frac{F(x+h_j e_j) - F(x-h_j e_j)}{2 h_j},
$$

which costs $2n$ evaluations of $F$ (the baseline $F(x)$ is *not* reused, unlike the one-sided forward-difference version). In exchange for the extra evaluations, the central estimate has $O(h_j^2)$ truncation error instead of $O(h_j)$, so it is markedly more accurate for the same step. It is also more error-prone to implement correctly: a column update must combine *two* perturbed evaluations ($F_+$ and $F_-$) rather than one, and it is easy to accidentally reuse a single baseline $F(x)$ (which silently degrades the method back to a one-sided estimate) or to mismatch the sign/order of the two evaluations in the numerator.

**Why the step must be scaled.** A naive implementation uses the *same* fixed absolute step $\epsilon$ for every coordinate, i.e. $h_j = \epsilon$ regardless of $x_j$. That works fine when $|x_j|$ is small or moderate, but silently fails when $|x_j|$ is large: double-precision floats carry roughly 15-17 significant decimal digits, so once

$$
|x_j| \;\gtrsim\; \frac{\epsilon}{\text{machine epsilon}} \;\approx\; \epsilon \times 10^{16},
$$

the expression `x_j + eps` rounds *exactly back to* `x_j` in floating point --- the perturbation is below the representable precision at that magnitude, so `F(x_plus)` and `F(x_minus)` are computed at the *same* point and the resulting column of $J$ comes out as (an inaccurate) zero, no matter how smooth $F$ actually is. There is no exception raised; the result is just silently wrong. The fix is to scale the step by the coordinate's own magnitude,

$$
h_j \;=\; \epsilon \cdot \max\!\bigl(1,\ |x_j|\bigr),
$$

so that the *relative* size of the perturbation stays roughly constant ($\approx \epsilon$) instead of the absolute size. This is the same idea used by production finite-difference codes (e.g. `scipy.optimize.approx_derivative`'s default relative step). With this scaling, `finite_difference_jacobian` remains accurate at $x_j$ near $10^{12}$ or beyond, not just near $x_j \approx O(1)$.

**Worked example (to fix the row/column convention).** For $F(x) = (x_1^2,\ x_1 x_2)$ at $x=(2,3)$: since $|x_j| > 1$ for both $j$, the per-coordinate steps are $h_1 = 2\epsilon$ and $h_2 = 3\epsilon$ (each scaled by its own coordinate magnitude, not by a single common $\epsilon$). Perturbing $x_1$ to $2 \pm 2\epsilon$ gives

$$
F_1(x_+)-F_1(x_-) \;\approx\; 2\,(2x_1)(2\epsilon) \;=\; 16\epsilon, \qquad
F_2(x_+)-F_2(x_-) \;\approx\; 2x_2(2\epsilon) \;=\; 12\epsilon,
$$

so dividing by $2h_1 = 4\epsilon$, **column 1** of $J$ is $(4,\ 3)^T$. Perturbing only $x_2$ gives $F_2(x_+)-F_2(x_-) \approx 2x_1(3\epsilon) = 12\epsilon$ (and $F_1$ unchanged), so dividing by $2h_2 = 6\epsilon$, **column 2** is $(0,\ 2)^T$. Putting the columns side by side,

$$
J(2,3) \;=\;
\begin{pmatrix}
4 & 0 \\[2pt]
3 & 2
\end{pmatrix},
$$

i.e. row $i$ always corresponds to output $F_i$, and column $j$ always corresponds to which input coordinate $x_j$ was perturbed (in both directions, with a step scaled to $x_j$'s own magnitude) to produce that column.

**Empty input.** When $x$ has shape $(0,)$, $F(x)$ is called once to determine $m$, and the returned array has shape $(m, 0)$ --- no columns, but $m$ rows. This is the natural limit of the $m \times n$ case as $n \to 0$, and it makes the function well-defined on $\mathbb{R}^0$ rather than silently returning `None` or raising an unrelated error.

## Testing Template

In [ ]:
# ============ TEST CASES ============

import numpy as np


def test_case_1():
    """Tier 1: diagonal F, generous tolerance."""
    F = lambda x: np.array([x[0]**2 - 1, x[1]**2 - 4])
    J = finite_difference_jacobian(F, np.array([1.0, 2.0]))
    assert np.allclose(J, [[2.0, 0.0], [0.0, 4.0]], atol=5e-2)
    print('test_case_1 passed')


def test_case_2():
    """Tier 2: affine F -- Jacobian is constant, exact for any correct method."""
    A = np.array([[2.0, 1.0], [0.0, 3.0]])
    b = np.array([1.0, -1.0])
    F = lambda x: A @ np.asarray(x) - b
    J = finite_difference_jacobian(F, np.array([5.0, -2.0]))
    assert np.allclose(J, A, atol=1e-4)
    print('test_case_2 passed')


def test_case_3():
    F = lambda x: np.array([x[0]*x[1], x[0] + x[1]**2])
    J = finite_difference_jacobian(F, np.array([2.0, 3.0]))
    expected = np.array([[3.0, 2.0], [1.0, 6.0]])
    assert np.allclose(J, expected, atol=5e-2)
    print('test_case_3 passed')


def test_case_4():
    F = lambda x: np.array([x[0]**3 - 2*x[0]])
    x = np.array([1.5])
    J = finite_difference_jacobian(F, x)
    true_J = np.array([[3*x[0]**2 - 2]])   # = 4.75
    assert np.allclose(J, true_J, atol=1e-8, rtol=0)
    print('test_case_4 passed')


def test_case_5():
    """Tier 5: same tight tolerance, 2x2 case to rule out n=1 accidents."""
    F = lambda x: np.array([x[0]**3 - x[1]**2, x[0]*x[1]**3])
    x = np.array([1.3, -0.7])
    J = finite_difference_jacobian(F, x)
    true_J = np.array([
        [3*x[0]**2, -2*x[1]],
        [x[1]**3, 3*x[0]*x[1]**2],
    ])
    assert np.allclose(J, true_J, atol=1e-8, rtol=0)
    print('test_case_5 passed')


def test_case_6():
    F = lambda x: np.array([5.0, -3.0])
    J = finite_difference_jacobian(F, np.array([]))
    assert J is not None, 'expected an array of shape (2, 0), got None'
    J = np.asarray(J)
    assert J.shape == (2, 0), f'expected shape (2, 0), got {J.shape}'
    print('test_case_6 passed')


def test_case_8():
    F = lambda x: np.array([x[0]**2])
    x = np.array([1e12])
    J = finite_difference_jacobian(F, x)
    true_J = np.array([[2e12]])
    assert np.allclose(J, true_J, rtol=5e-10), (
        f"step scaling missing at large |x|: got {J}, expected {true_J}"
    )
    print('test_case_8 passed')


def test_case_9():
    """x must be 1-D. A 2-D array is not a point in R^n and must be
    rejected rather than silently flattened or misinterpreted."""
    F = lambda x: np.array([x[0]**2])
    x_bad = np.array([[1.0, 2.0], [3.0, 4.0]])
    try:
        finite_difference_jacobian(F, x_bad)
        assert False, 'expected ValueError for 2-D x'
    except ValueError:
        pass
    print('test_case_9 passed')


def test_case_11():
    F = lambda x: np.array([x[0], x[1], x[0] + x[1]])
    x = np.array([0.5, -0.7])
    J = finite_difference_jacobian(F, x)
    expected = np.array([
        [1.0, 0.0],
        [0.0, 1.0],
        [1.0, 1.0],
    ])
    assert J.shape == (3, 2), f'expected shape (3, 2), got {J.shape}'
    assert np.allclose(J, expected, atol=1e-6), f'expected {expected}, got {J}'
    print('test_case_11 passed')

## Solution

In [ ]:
import numpy as np
from typing import Callable


def finite_difference_jacobian(F: Callable, x: np.ndarray, eps: float = 1e-6) -> np.ndarray:
    """Ordinary central-difference Jacobian of F at x, with per-coordinate
    step h_j = eps * max(1, |x_j|) so the perturbation stays resolvable
    for large |x_j|."""
    if eps <= 0:
        raise ValueError("eps must be positive")

    x = np.asarray(x, dtype=float)
    if x.ndim != 1:
        raise ValueError("x must be a 1-D array")

    F0 = np.asarray(F(x), dtype=float)
    if F0.ndim != 1:
        raise ValueError("F(x) must return a 1-D array")

    n = len(x)
    m = len(F0)
    J = np.zeros((m, n))

    for j in range(n):
        h = eps * max(1.0, abs(x[j]))
        x_plus = x.copy(); x_plus[j] += h
        x_minus = x.copy(); x_minus[j] -= h
        F_plus = np.asarray(F(x_plus), dtype=float)
        F_minus = np.asarray(F(x_minus), dtype=float)
        J[:, j] = (F_plus - F_minus) / (2 * h)

    return J

# Subproblem 2

## Prompt

Implement the Newton step: given the Jacobian and function value at the current point, solve the linear system for the update that would zero a locally linear model of $F$.

```python
def newton_step(J: np.ndarray, Fx: np.ndarray, lam: float = 0.0) -> np.ndarray:
    """
    Solve the (possibly damped) Newton step.

    lam == 0 (default). Undamped Gauss-Newton step. Solves J @ delta = -Fx
    in the least-squares sense. Only square or overdetermined systems
    (m >= n) are supported. Singular or ill-conditioned J (condition
    number > 1e10) is rejected.

    lam > 0. Levenberg-Marquardt damped step. Solves the damped normal
    equations

        (J.T @ J + lam * I) @ delta = -J.T @ Fx,

    with left-hand side symmetric positive definite for any J. This path
    accepts any shape (m, n), including underdetermined (m < n), singular,
    and ill-conditioned J.

    Parameters
    ----------
    J : ndarray, shape (m, n)
        Jacobian at the current point.
    Fx : ndarray, shape (m,)
        Function (residual) value at the current point.
    lam : float
        Non-negative Levenberg-Marquardt damping parameter. Default 0
        (undamped Gauss-Newton). Must be >= 0.

    Returns
    -------
    delta : ndarray, shape (n,)
        Step such that J @ delta = -Fx (when lam == 0 and m == n), or such
        that (J.T @ J + lam * I) @ delta = -J.T @ Fx (when lam > 0).

    Raises
    ------
    ValueError
        If lam < 0. If J or Fx contain any non-finite (NaN or Inf) entries.
        If J is not 2-D, if Fx is not 1-D, or if J.shape[0] != len(Fx).
        At lam == 0 only: if m < n, or if J is singular or too
        ill-conditioned (condition number > 1e10).
    """
    ...
    return delta
```


## Background

Near a point $x$, the first-order Taylor expansion of $F$ is $F(x+\delta) \approx F(x) + J(x)\delta$. Newton's method chooses $\delta$ to make this local linear model exactly zero:

$$
F(x) + J(x)\,\delta = 0 \quad\Longrightarrow\quad J(x)\,\delta = -F(x).
$$

This is a square linear system, solved directly rather than by explicitly forming $J^{-1}$, which is both cheaper and more numerically stable. Adding $\delta$ to $x$ gives the next Newton iterate; when $x$ is already close to a root and $F$ is smooth, this converges quadratically.

**Singular and ill-conditioned Jacobians.** If $J$ is exactly singular, `numpy.linalg.solve` already raises on its own. But a matrix can also be *numerically* singular without being *exactly* singular in floating point: if $J$'s condition number is astronomically large (say $10^{15}$), `numpy.linalg.solve` will still happily return a "solution", but that vector is dominated by rounding error and is numerically meaningless as a Newton step --- floating point does not raise an error just because a result is garbage. A robust implementation therefore needs to explicitly check the condition number (via `numpy.linalg.cond`) and treat anything above a practical threshold as if it were exactly singular. This task's numerical-sensitivity policy sets that threshold at $10^{10}$: below it, the conditioning is close enough to well-posed that the computed step is trustworthy for the purposes of Newton's method; above it, the reported step is treated as unreliable and rejected by `ValueError`. This threshold is a policy choice for this task, not a mathematical theorem that every result above $10^{10}$ is meaningless.

**Precision versus dynamic range.** Double-precision `float64` carries about 15-17 significant decimal digits (its *precision*) and separately represents magnitudes from roughly $10^{-308}$ to $10^{308}$ (its *dynamic range*). NumPy documents these as independent properties. The $10^{10}$ condition-number cutoff lives well inside both bounds --- it is a practical reliability threshold, not a bound imposed by either precision or dynamic range.

**Non-finite inputs need the same explicit treatment.** If a caller passes a `J` or `Fx` that already contains a `NaN` or `Inf` (for example because an earlier finite-difference evaluation diverged), `numpy.linalg.cond` and `numpy.linalg.solve` do **not** reliably raise: `cond` on a matrix containing `NaN` typically returns `NaN` itself (which fails a $> 10^{10}$ comparison silently, since any comparison against `NaN` is `False`), and `solve` on such inputs simply propagates the `NaN`/`Inf` into $\delta$ rather than raising. The finiteness of $J$ and $Fx$ must therefore be checked *before* computing the condition number or calling `solve`, not inferred afterward from whether an exception happened to occur.

**Levenberg--Marquardt damping.** When the Jacobian is singular, underdetermined, or too ill-conditioned for the plain square system to be meaningful, a damped step is used instead. The Levenberg--Marquardt modification solves the **damped normal equations**

$$
\bigl(J^{T} J + \lambda I\bigr)\,\delta \;=\; -J^{T} F(x),
$$

where $\lambda \ge 0$ is the damping parameter and $I$ is the $n \times n$ identity. For $\lambda > 0$ the left-hand matrix is symmetric positive definite regardless of $J$'s rank or shape, so `numpy.linalg.solve` always returns a well-defined step. As $\lambda \to 0$ the damped step approaches the least-squares Gauss--Newton step; as $\lambda \to \infty$ the step is driven toward zero, so larger $\lambda$ gives a shorter, more conservative update. This path also permits **underdetermined** systems ($m < n$), which the undamped path rejects.

**Why the damping must be applied to $J^{T} J$, not to $J$ directly.** A common but incorrect variant forms

$$
\bigl(J + \lambda I\bigr)\,\delta \;=\; -F(x)
$$

by analogy with Tikhonov-style regularization of the original system. That is a different equation: it adds $\lambda$ to the diagonal of $J$ rather than to the diagonal of $J^{T} J$, and the resulting step is not the Levenberg--Marquardt step. A nonidentity test at $\lambda > 0$ is therefore needed to distinguish the two variants --- the identity case $J = I$ happens to give the same step under both, because

$$
J^{T} J + \lambda I \;=\; I + \lambda I \;=\; (1+\lambda) I \;=\; J + \lambda I
$$

coincide when $J = I$.

## Testing Template

In [ ]:
# ============ TEST CASES ============

import numpy as np


def test_case_1():
    """Diagonal Jacobian."""
    J = np.array([[2.0, 0.0], [0.0, 4.0]])
    Fx = np.array([0.5, 1.0])
    delta = newton_step(J, Fx)
    assert np.allclose(J @ delta, -Fx, atol=1e-10)
    print('test_case_1 passed')


def test_case_2():
    """Dense, well-conditioned Jacobian."""
    J = np.array([[1.0, 2.0], [3.0, 4.0]])
    Fx = np.array([5.0, 6.0])
    delta = newton_step(J, Fx)
    assert np.allclose(J @ delta + Fx, 0.0, atol=1e-10)
    print('test_case_2 passed')


def test_case_3():
    J = np.array([[1.0, 1.0], [1.0, 1.0 + 1e-15]])
    Fx = np.array([1.0, 1.0])
    try:
        newton_step(J, Fx)
        assert False, 'expected ValueError for severely ill-conditioned Jacobian'
    except ValueError:
        pass
    print('test_case_3 passed')


def test_case_4():
    """Moderately ill-conditioned (cond ~ 1e6): must be accepted."""
    J = np.array([[1e6, 0.0], [0.0, 1.0]])
    Fx = np.array([2.0, 3.0])
    delta = newton_step(J, Fx)
    assert np.allclose(J @ delta, -Fx, atol=1e-6)
    print('test_case_4 passed')


def test_case_5():
    """Singular Jacobian: must be rejected."""
    J = np.array([[1.0, 2.0], [2.0, 4.0]])
    Fx = np.array([1.0, 2.0])
    try:
        newton_step(J, Fx)
        assert False, 'expected ValueError for singular Jacobian'
    except ValueError:
        pass
    print('test_case_5 passed')


def test_case_6():
    """Underdetermined system (m < n) at lam=0: must be rejected."""
    J = np.array([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
    Fx = np.array([1.0, 2.0])
    try:
        newton_step(J, Fx)
        assert False, 'expected ValueError for underdetermined Jacobian'
    except ValueError:
        pass
    print('test_case_6 passed')


def test_case_7():
    """Mismatched dimensions."""
    J = np.array([[1.0, 2.0], [3.0, 4.0]])
    Fx = np.array([1.0, 2.0, 3.0])
    try:
        newton_step(J, Fx)
        assert False, 'expected ValueError for mismatched dimensions'
    except ValueError:
        pass
    print('test_case_7 passed')


def test_case_8():
    """Identity matrix: delta == -Fx."""
    J = np.eye(3)
    Fx = np.array([1.0, -2.0, 3.0])
    delta = newton_step(J, Fx)
    assert np.allclose(delta, [-1.0, 2.0, -3.0])
    print('test_case_8 passed')


def test_case_9():
    """Non-finite entries in J and in Fx must be rejected explicitly."""
    J_nan = np.array([[1.0, 0.0], [0.0, np.nan]])
    Fx_ok = np.array([1.0, 2.0])
    try:
        newton_step(J_nan, Fx_ok)
        assert False, 'expected ValueError for non-finite entry in J'
    except ValueError:
        pass

    J_ok = np.array([[1.0, 0.0], [0.0, 1.0]])
    Fx_inf = np.array([1.0, np.inf])
    try:
        newton_step(J_ok, Fx_inf)
        assert False, 'expected ValueError for non-finite entry in Fx'
    except ValueError:
        pass
    print('test_case_9 passed')


def test_case_10():
    """cond(J) just BELOW the 1e10 threshold must be ACCEPTED."""
    J = np.array([[1e9, 0.0], [0.0, 1.0]])
    Fx = np.array([1e9, 1.0])
    delta = newton_step(J, Fx)
    assert np.allclose(J @ delta, -Fx, atol=1e-6), (
        f"cond ~ 1e9 is below the 1e10 threshold and must be accepted, "
        f"got delta={delta}"
    )
    print('test_case_10 passed')


def test_case_11():
    """cond(J) just ABOVE the 1e10 threshold must be REJECTED."""
    J = np.array([[1e11, 0.0], [0.0, 1.0]])
    Fx = np.array([1.0, 1.0])
    try:
        newton_step(J, Fx)
        assert False, 'expected ValueError for cond ~ 1e11 > 1e10 threshold'
    except ValueError:
        pass
    print('test_case_11 passed')


def test_case_12():
    """Damping is applied: lam=1 on J=I gives delta = -Fx / (1 + lam)."""
    J = np.eye(2)
    Fx = np.array([1.0, 1.0])
    delta = newton_step(J, Fx, lam=1.0)
    expected = np.array([-0.5, -0.5])
    assert np.allclose(delta, expected, atol=1e-12), (
        f"lam=1.0 should regularize the step; got {delta}, expected {expected}. "
        f"A result of [-1, -1] indicates lam was silently ignored."
    )
    print('test_case_12 passed')


def test_case_13():
    J = np.array([[2.0, 1.0], [1.0, 3.0]])
    Fx = np.array([1.0, 1.0])
    delta = newton_step(J, Fx, lam=1.0)
    expected = np.array([-13.0 / 41.0, -9.0 / 41.0])
    assert np.allclose(delta, expected, atol=1e-12), (
        f"damped step mismatch: got {delta}, expected {expected}"
    )
    print('test_case_13 passed')

## Solution

In [ ]:
import numpy as np
from typing import Callable, Tuple


def newton_step(J: np.ndarray, Fx: np.ndarray, lam: float = 0.0) -> np.ndarray:
    """
    Levenberg-Marquardt damped Gauss-Newton step.

    lam == 0: exact solve for square J, least-squares for overdetermined
    J (m > n); underdetermined (m < n) and singular/ill-conditioned J are
    rejected.

    lam > 0: solves the damped normal equations
    (J.T @ J + lam * I) @ delta = -J.T @ Fx, which is always symmetric
    positive definite regardless of J's rank or shape, so it accepts
    singular, ill-conditioned, and underdetermined J as well.

    Parameters
    ----------
    J : ndarray, shape (m, n)
        Jacobian at the current point.
    Fx : ndarray, shape (m,)
        Function (residual) value at the current point.
    lam : float
        Levenberg-Marquardt damping parameter (must be >= 0).

    Returns
    -------
    delta : ndarray, shape (n,)
        The (possibly damped) Gauss-Newton step.

    Raises
    ------
    ValueError
        If lam < 0. If J or Fx contain any non-finite (NaN or Inf)
        entries. If J's shape is incompatible with Fx. At lam == 0 only:
        if the system is underdetermined (m < n), or if J is singular or
        so ill-conditioned (condition number greater than 1e10) that the
        computed step would be dominated by floating-point rounding error.
    """
    J = np.asarray(J, dtype=float)
    Fx = np.asarray(Fx, dtype=float)

    # Non-finite entries must be rejected explicitly: cond()/lstsq()/solve()
    # do not reliably raise on NaN/Inf -- they silently produce garbage.
    if not np.all(np.isfinite(J)):
        raise ValueError("Jacobian contains non-finite (NaN or Inf) entries")
    if not np.all(np.isfinite(Fx)):
        raise ValueError("Fx contains non-finite (NaN or Inf) entries")

    if J.ndim != 2 or Fx.ndim != 1:
        raise ValueError("J must be 2-D and Fx must be 1-D")

    m, n = J.shape
    if m != len(Fx):
        raise ValueError("Jacobian and Fx dimensions must match")

    if lam < 0:
        raise ValueError("lam must be non-negative")

    if lam == 0.0:
        # Plain Gauss-Newton path: same restrictions as before.
        if m < n:
            raise ValueError(
                "underdetermined system (fewer equations than unknowns) is "
                "not supported at lam=0; pass lam > 0 to regularize it"
            )
        cond = np.linalg.cond(J)
        if cond > 1e10:
            raise ValueError(
                f"Jacobian is too ill-conditioned "
                f"(condition number = {cond:.2e} > 1e10)"
            )
        delta, _residuals, _rank, _sv = np.linalg.lstsq(J, -Fx, rcond=None)
    else:
        # Damped normal equations: J.T @ J + lam * I is always symmetric
        # positive definite for lam > 0, regardless of J's rank or shape,
        # so this is well-defined even for singular/ill-conditioned/
        # underdetermined J.
        A = J.T @ J + lam * np.eye(n)
        b = -(J.T @ Fx)
        delta = np.linalg.solve(A, b)

    if not np.all(np.isfinite(delta)):
        raise ValueError("Newton step contains non-finite values")

    return delta

# Main Problem

## Prompt

import numpy as np
from typing import Callable, Tuple


```python
import numpy as np
from typing import Callable, Tuple


def finite_difference_jacobian(F: Callable, x: np.ndarray, eps: float = 1e-6) -> np.ndarray:
    """
    Central-difference approximation of the Jacobian of F at x.

    Same contract as Subproblem 1: ordinary central differences with
    per-coordinate step h_j = eps * max(1, |x_j|). Returns shape (m, n),
    where m = len(F(x)).

    Parameters
    ----------
    F : callable
        F(x) -> ndarray, shape (m,).
    x : ndarray, shape (n,)
    eps : float
        Relative finite-difference step. Must be > 0.

    Returns
    -------
    J : ndarray, shape (m, n)
        J[i, j] = dF_i/dx_j.

    Raises
    ------
    ValueError
        If eps <= 0, if x is not 1-D, or if F(x) does not return a 1-D array.
    """
    ...


def newton_step(J: np.ndarray, Fx: np.ndarray, lam: float = 0.0) -> np.ndarray:
    """
    (Possibly damped) Newton step. Same contract as Subproblem 2.

    lam == 0: undamped Gauss-Newton step (least-squares solve of J @ delta
    = -Fx). Rejects underdetermined (m < n), singular, and too
    ill-conditioned (cond > 1e10) Jacobians.

    lam > 0: Levenberg-Marquardt damped step, solving
    (J.T @ J + lam * I) @ delta = -J.T @ Fx. Accepts any shape (m, n).

    Parameters
    ----------
    J : ndarray, shape (m, n)
    Fx : ndarray, shape (m,)
    lam : float
        Non-negative damping parameter. Default 0 (undamped).

    Returns
    -------
    delta : ndarray, shape (n,)

    Raises
    ------
    ValueError
        If lam < 0, if J or Fx contain non-finite entries, if J or Fx
        shapes are incompatible, or (at lam == 0 only) if the system is
        underdetermined or too ill-conditioned.
    """
    ...


def main_problem(F: Callable, x0: np.ndarray, tol: float = 1e-10,
                 max_iter: int = 100) -> Tuple[np.ndarray, int]:
    """
    Find a root of F(x) = 0 using Newton's method.

    The main solver expects a square system: when x has shape (n,),
    F(x) must return shape (n,). (Rectangular Jacobians are supported
    by finite_difference_jacobian in Subproblem 1, but the main solver
    uses only the square Newton step at lam=0.)

    Parameters
    ----------
    F : callable
        F(x) -> ndarray, shape (n,). Must be finite-valued along the
        iterates.
    x0 : ndarray, shape (n,)
        Initial guess. The returned array is a fresh copy and does not
        alias x0, so mutating it will not affect the caller's input.
    tol : float
        Convergence tolerance on ||F(x)||. Must be positive.
    max_iter : int
        Maximum number of Newton iterations. Must be positive.

    Returns
    -------
    x : ndarray, shape (n,)
        A finite iterate with ||F(x)|| < tol. Freshly allocated, not a
        view of x0.
    n_iter : int
        Number of Newton updates performed. If x0 already satisfies
        ||F(x0)|| < tol, n_iter == 0. If one update is enough (e.g. affine
        F), n_iter == 1.

    Raises
    ------
    ValueError
        If tol <= 0 or max_iter <= 0. If Newton's method produces a
        non-finite iterate or a non-finite F(x) value. If the final
        iterate does not satisfy ||F(x)|| < tol after max_iter updates.
    """
    ...
```


def main_problem(F: Callable, x0: np.ndarray, tol: float = 1e-10,
                 max_iter: int = 100) -> Tuple[np.ndarray, int]:
    """
    Find a root of F(x) = 0 using Newton's method.

    Parameters
    ----------
    F : callable
        F(x) -> ndarray, shape (m,). Differentiable near the root.
    x0 : ndarray, shape (n,)
        Initial guess.
    tol : float
        Convergence tolerance on ||F(x)||. Must be positive.
    max_iter : int
        Maximum number of Newton iterations. Must be positive.

    Returns
    -------
    x : ndarray, shape (n,)
        Final iterate.
    n_iter : int
        Number of Newton iterations performed.

    Raises
    ------
    ValueError
        If tol <= 0 or max_iter <= 0.
    """
    ...

## Background

Starting from $x_0$, Newton's method repeats:

1. Evaluate $F(x_k)$; stop if $\|F(x_k)\|$ is already below `tol`.
2. Estimate the Jacobian $J(x_k)$ with `finite_difference_jacobian`.
3. Solve for the step with `newton_step`, giving $\delta_k$ such that $J(x_k)\delta_k = -F(x_k)$.
4. Update $x_{k+1} = x_k + \delta_k$.

For $F$ that are already affine ($F(x) = Ax - b$), the Jacobian is constant and equal to $A$ everywhere, so the linear model is exact and Newton's method converges in a **single** update regardless of the starting point, giving $n_{\text{iter}} = 1$ for any $x_0$ not already a root. For smooth nonlinear $F$ with a nonsingular Jacobian at the root, convergence is locally quadratic: once $x_k$ is close enough, the number of correct digits roughly doubles every iteration.

**Finiteness.** Even when convergence is not guaranteed, every iterate that the solver returns must be finite and must have $\|F(x)\| < \text{tol}$. A naive implementation can silently produce a `NaN` step when $F$ overflows at a huge iterate: for example, Newton on $F(x) = e^{x} - 2$ from $x_0 = -10$ jumps to $x_1 \approx 4.4\times 10^{4}$, at which $e^{x_1}$ overflows to `Inf`; the next Newton step then evaluates $\text{Inf}/\text{Inf}$ and returns `NaN`. Because `NaN < tol` is `False`, a naive convergence check accepts this and returns `(nan, ...)`. A correct implementation therefore checks that both $F(x)$ and the iterate itself are finite at every step, and raises `ValueError` on the first non-finite value rather than continuing.

**No aliasing.** The returned array must be a fresh array, not a view into the caller's `x0`. If the solver returns `x0` unchanged (because `||F(x0)||` was already below `tol`), a subsequent in-place mutation of the returned array would silently corrupt the caller's input — the test suite's no-alias regression checks this.

**Non-convergence.** Newton's method is only a local method. A poor starting point can make it diverge or cycle forever without ever approaching a root. A classical example is $f(x) = x^3 - 2x + 2$ started at $x_0 = 0$: the iteration maps $0 \mapsto 1 \mapsto 0 \mapsto 1 \mapsto \cdots$ indefinitely, never getting closer to the true (irrational) root near $x \approx -1.77$. Because of this, an implementation must not silently return a stale, non-converged $x$ after exhausting `max_iter` — it must check whether convergence was actually achieved and signal the failure via `ValueError`.

## Testing Template

In [ ]:
# ============ TEST CASES ============

import numpy as np


def test_case_1():
    """Standard 2x2 nonlinear system; generous tolerance on ||F||."""
    F = lambda x: np.array([x[0]**2 + x[1]**2 - 1, x[0] - x[1]])
    x, n_iter = main_problem(F, np.array([1.0, 0.5]))
    assert isinstance(n_iter, (int, np.integer))
    assert np.allclose(x, [1/np.sqrt(2), 1/np.sqrt(2)], atol=1e-4)
    assert np.allclose(F(x), [0.0, 0.0], atol=5e-2)
    print('test_case_1 passed')


def test_case_2():
    """Another 2x2 nonlinear system; tight tolerance on ||F||."""
    F = lambda x: np.array([x[0]**3 - x[1] - 1, x[0] + x[1]**2 - 3])
    x, n_iter = main_problem(F, np.array([1.5, 1.0]))
    assert isinstance(n_iter, (int, np.integer))
    assert np.allclose(F(x), [0.0, 0.0], atol=1e-8)
    print('test_case_2 passed')


def test_case_3():
    """Affine F: Newton should converge in a small number of steps."""
    A = np.array([[2.0, 1.0], [1.0, 3.0]])
    b = np.array([1.0, -1.0])
    F = lambda x: A @ x - b
    x, n_iter = main_problem(F, np.array([100.0, -50.0]), tol=1e-10, max_iter=10)
    true_root = np.linalg.solve(A, b)
    assert np.allclose(x, true_root, atol=1e-10)
    assert n_iter <= 3, f"affine F should converge fast, got {n_iter}"
    print('test_case_3 passed')


def test_case_4():
    """tol must be positive; then a standard scalar convergence check."""
    F = lambda x: np.array([x[0]**2 - 1])
    for bad in [0.0, -1.0]:
        try:
            main_problem(F, np.array([1.0]), tol=bad)
            assert False, f'expected ValueError for tol={bad}'
        except ValueError:
            pass
    x, n_iter = main_problem(F, np.array([0.5]))
    assert np.allclose(x, [1.0], atol=1e-10)
    print('test_case_4 passed')


def test_case_5():
    """Decoupled 2x2 system: two independent scalar roots."""
    F = lambda x: np.array([x[0]**2 - 1, x[1]**2 - 1])
    x, n_iter = main_problem(F, np.array([2.0, 2.0]))
    assert np.allclose(x, [1.0, 1.0], atol=1e-10)
    print('test_case_5 passed')


def test_case_6():
    A = np.array([[2.0, 1.0], [1.0, 3.0]])
    b = np.array([1.0, -1.0])
    F = lambda x: A @ x - b
    x, n_iter = main_problem(F, np.array([100.0, -50.0]),
                             tol=1e-6, max_iter=10)
    assert n_iter == 1, f"expected n_iter == 1, got {n_iter}"
    # Sanity: the returned iterate really is the affine root.
    true_root = np.linalg.solve(A, b)
    assert np.allclose(x, true_root, atol=1e-6)
    print('test_case_6 passed')


def test_case_7():
    """Jacobian accuracy at default eps, in line with Subproblem 1."""
    F = lambda x: np.array([x[0]**3 - 2*x[0]])
    x = np.array([1.5])
    J = finite_difference_jacobian(F, x)
    true_J = np.array([[3*x[0]**2 - 2]])      # = 4.75
    assert np.allclose(J, true_J, atol=1e-8, rtol=0), (
        f"Jacobian not accurate enough: got {J}, expected {true_J}"
    )

    F2 = lambda x: np.array([x[0]**2])
    x2 = np.array([1e12])
    J2 = finite_difference_jacobian(F2, x2)
    true_J2 = np.array([[2e12]])
    assert np.allclose(J2, true_J2, rtol=5e-10), (
        f"step scaling missing at large |x|: got {J2}, expected {true_J2}"
    )
    print('test_case_7 passed')


def test_case_8():
    """max_iter must be validated like tol: non-positive values raise."""
    F = lambda x: np.array([x[0]**2 - 1])
    for bad in [0, -1, -100]:
        try:
            main_problem(F, np.array([0.5]), max_iter=bad)
            assert False, f'expected ValueError for max_iter={bad}'
        except ValueError:
            pass
    print('test_case_8 passed')


def test_case_9():
    """Newton from a very large initial guess on x^2 - 4 = 0."""
    F = lambda x: np.array([x[0]**2 - 4.0])
    x, n_iter = main_problem(F, np.array([1e12]), tol=1e-8, max_iter=100)
    x_arr = np.asarray(x)
    assert x_arr.size == 1, f"expected scalar output, got shape {x_arr.shape}"
    xv = float(x_arr[0])
    assert abs(abs(xv) - 2.0) < 1e-4, \
        f"Newton from x0=1e12 failed to reach a root of x^2-4; got x={xv}"
    print('test_case_9 passed')


def test_case_10():
    """Non-convergence must raise ValueError, not silently return x0.

    Newton on f(x) = x^3 - 2x + 2 from x0 = 0 cycles 0 -> 1 -> 0 -> 1...
    forever, never approaching the true root near x ~= -1.77.
    """
    F = lambda x: np.array([x[0]**3 - 2*x[0] + 2])
    try:
        main_problem(F, np.array([0.0]), tol=1e-10, max_iter=50)
        assert False, (
            "expected ValueError: Newton on x^3-2x+2 from x0=0 cycles "
            "between 0 and 1 and never converges"
        )
    except ValueError:
        pass
    print('test_case_10 passed')


def test_case_11():
    """main_problem must not return the same object as x0 (no aliasing)."""
    F = lambda x: np.array([x[0] - 5.0])
    x0 = np.array([5.0])
    x, n_iter = main_problem(F, x0, tol=1e-10, max_iter=10)
    assert n_iter == 0, f"expected immediate convergence, got n_iter={n_iter}"
    x[0] = 999.0
    assert x0[0] == 5.0, (
        "mutating the returned x changed x0; the returned array must "
        "not alias the caller's input"
    )
    print('test_case_11 passed')


def test_case_12():
    """finite_difference_jacobian(F, np.array([])) must return shape (m, 0),
    where m = len(F(x)). F: R^0 -> R^m is a valid zero-dimensional input;
    returning None or (0, 0) is a silent well-posedness gap, and crashing
    on it is worse."""
    F = lambda x: np.array([5.0, -3.0])
    J = finite_difference_jacobian(F, np.array([]))
    assert J is not None, "expected array of shape (2, 0), got None"
    J = np.asarray(J)
    assert J.shape == (2, 0), f"expected shape (2, 0), got {J.shape}"
    print('test_case_12 passed')


def test_case_13():
    """Non-finite iterates must raise ValueError, not be returned as a
    converged root.

    Newton on F(x) = exp(x) - 2 from x0 = -10: the first iterate is
    ~4.4e4, at which exp(4.4e4) overflows to Inf; the resulting Newton
    step is NaN. The correct behaviour is to raise ValueError, not to
    return (array([nan]), 10)."""
    F = lambda x: np.array([np.exp(x[0]) - 2.0])
    try:
        result = main_problem(F, np.array([-10.0]), tol=1e-10, max_iter=100)
        assert False, (
            f"expected ValueError for divergent Newton; got {result}"
        )
    except ValueError:
        pass
    print('test_case_13 passed')

## Solution

In [ ]:
import numpy as np
from typing import Callable, Tuple


def finite_difference_jacobian(F: Callable, x: np.ndarray, eps: float = 1e-6) -> np.ndarray:
    """Inlined from Subproblem 1: ordinary central differences with
    per-coordinate step h_j = eps * max(1, |x_j|)."""
    if eps <= 0:
        raise ValueError("eps must be positive")

    x = np.asarray(x, dtype=float)
    if x.ndim != 1:
        raise ValueError("x must be a 1-D array")

    F0 = np.asarray(F(x), dtype=float)
    if F0.ndim != 1:
        raise ValueError("F(x) must return a 1-D array")

    n = len(x)
    m = len(F0)
    J = np.zeros((m, n))

    for j in range(n):
        h = eps * max(1.0, abs(x[j]))
        x_plus = x.copy(); x_plus[j] += h
        x_minus = x.copy(); x_minus[j] -= h
        F_plus = np.asarray(F(x_plus), dtype=float)
        F_minus = np.asarray(F(x_minus), dtype=float)
        J[:, j] = (F_plus - F_minus) / (2 * h)

    return J


def newton_step(J: np.ndarray, Fx: np.ndarray, lam: float = 0.0) -> np.ndarray:
    """Inlined from Subproblem 2: Levenberg-Marquardt damped Newton step."""
    J = np.asarray(J, dtype=float)
    Fx = np.asarray(Fx, dtype=float)

    if not np.all(np.isfinite(J)):
        raise ValueError("Jacobian contains non-finite (NaN or Inf) entries")
    if not np.all(np.isfinite(Fx)):
        raise ValueError("Fx contains non-finite (NaN or Inf) entries")

    if J.ndim != 2 or Fx.ndim != 1:
        raise ValueError("J must be 2-D and Fx must be 1-D")

    m, n = J.shape
    if m != len(Fx):
        raise ValueError("Jacobian and Fx dimensions must match")

    if lam < 0:
        raise ValueError("lam must be non-negative")

    if lam == 0.0:
        if m < n:
            raise ValueError(
                "underdetermined system is not supported at lam=0; "
                "pass lam > 0 to regularize it"
            )
        cond = np.linalg.cond(J)
        if cond > 1e10:
            raise ValueError(
                f"Jacobian is too ill-conditioned "
                f"(condition number = {cond:.2e} > 1e10)"
            )
        delta, _residuals, _rank, _sv = np.linalg.lstsq(J, -Fx, rcond=None)
    else:
        A = J.T @ J + lam * np.eye(n)
        b = -(J.T @ Fx)
        delta = np.linalg.solve(A, b)

    if not np.all(np.isfinite(delta)):
        raise ValueError("Newton step contains non-finite values")

    return delta


def main_problem(F: Callable, x0: np.ndarray, tol: float = 1e-10,
                 max_iter: int = 100) -> Tuple[np.ndarray, int]:
    """
    Newton's method for a square system F: R^n -> R^n.

    Returns (x, n_iter), where x is a freshly allocated array with
    ||F(x)|| < tol and n_iter is the number of Newton updates taken.
    Raises ValueError if the iterates or F values become non-finite, or
    if convergence is not reached within max_iter updates.
    """
    if tol <= 0:
        raise ValueError("tol must be positive")
    if max_iter <= 0:
        raise ValueError("max_iter must be positive")

    # Fresh copy: the returned array never aliases x0.
    x = np.array(x0, dtype=float)

    for i in range(max_iter + 1):
        Fx = np.asarray(F(x), dtype=float)
        if not np.all(np.isfinite(Fx)):
            raise ValueError(
                "Newton's method encountered a non-finite F(x) value"
            )
        if np.linalg.norm(Fx) < tol:
            # Success: x is finite, ||F(x)|| is below tol.
            return x, i

        if i >= max_iter:
            break

        try:
            J = finite_difference_jacobian(F, x)
            delta = newton_step(J, Fx)
        except ValueError as exc:
            raise ValueError(
                f"Newton step failed at iteration {i}: {exc}"
            ) from exc

        x_new = x + delta
        if not np.all(np.isfinite(x_new)):
            raise ValueError(
                "Newton's method produced a non-finite iterate"
            )
        x = x_new

    raise ValueError(
        f"Newton's method did not converge within max_iter={max_iter} updates"
    )